# DATA PREPROCESSING
This notebook will capture the cleaning, feature engineering and feature selection part of the week 3 deliverable. The original dataset given in week 1 will be used.

## Objectives
The aim is to prepare the dataset for predictive modelling and churn analysis.


In [1]:
#reading the raw file
import pandas as pd
raw_path=r"Dataset.xlsx"
raw = pd.read_excel(raw_path)
#Display the first 2 rows of the dataframe
raw.head(2)


,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date
0,06/14/2023 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,06/29/2024 18:52:39,Faria,2001-12-01 00:00:00,Female,Pakistan,Nwihs,Radiology,2024-11-03 12:01:41,Started,1080,06/14/2023 12:36:09,2022-03-11 18:30:39
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,06/29/2024 18:52:39,Poojitha,08/16/2000,Female,India,SAINT LOUIS,Information Systems,2024-11-03 12:01:41,Started,1080,2023-01-05 06:08:21,2022-03-11 18:30:39


In [2]:
#get the size of dataset
raw.shape

(8558, 16)

- check the datatypes of each column

In [3]:
#Display the datatypes of each column in the dataframe
raw.dtypes

Learner SignUp DateTime            object
Opportunity Id                     object
Opportunity Name                   object
Opportunity Category               object
Opportunity End Date               object
First Name                         object
Date of Birth                      object
Gender                             object
Country                            object
Institution Name                   object
Current/Intended Major             object
Entry created at           datetime64[ns]
Status Description                 object
Status Code                         int64
Apply Date                         object
Opportunity Start Date             object
dtype: object

**Data Cleaning**

In [4]:
# Check data quality issues
print("Data Quality Issues:")
print(f"Missing values:\n{raw.isnull().sum()}")
print(f"\nDuplicate rows: {raw.duplicated().sum()}")

# Check for inconsistent categorical values
print(f"\nUnique values in categorical columns:")
categorical_cols = ['Opportunity Name','Gender', 'Country','Opportunity Category','Institution Name', 'Current/Intended Major', 'Status Description','Status Code']
for col in categorical_cols:
    if col in raw.columns:
        print(f"{col}: {raw[col].unique()}")


Data Quality Issues:
Missing values:
Learner SignUp DateTime       0
Opportunity Id                0
Opportunity Name              0
Opportunity Category          0
Opportunity End Date          0
First Name                    0
Date of Birth                 0
Gender                        0
Country                       0
Institution Name              5
Current/Intended Major        5
Entry created at              0
Status Description            0
Status Code                   0
Apply Date                    0
Opportunity Start Date     3794
dtype: int64

Duplicate rows: 0

Unique values in categorical columns:
Opportunity Name: ['Career Essentials: Getting Started with Your Professional Journey'
 'Slide Geeks: A Presentation Design Competition' 'Digital Marketing'
 'Health Care Management' 'Innovation & Entrepreneurship'
 'Project Management' 'Data Visualization' 'CPR/AED Certification'
 'Mental and Physical Health Session'
 'Jump Start: Developing your Emotional Intelligence'
 'Join

In [5]:
raw['Status Description'].value_counts()

Status Description
Rejected          3569
Team Allocated    3276
Started            767
Dropped Out        617
Waitlisted         109
Applied            105
Withdraw            86
Rewards Award       29
Name: count, dtype: int64

***Handling missing values***
- For Institution name and Current/Intended Major rows with missing values are dropped
- For Opportunity Start Date having 3794 empty rows are replaced with a place holder

In [6]:
#inputting unknown is some missing values (handling missing data)
raw['Opportunity Start Date']=raw['Opportunity Start Date'].fillna('unknown')
#Drop rows with null values from the dataframe (handling missing data)
data=raw.dropna()
#get the size to ensure accuracy
data.shape

(8548, 16)

***Handling columns formats***
- The dataset contain categorical columns and date columns

In [7]:
#converting all columns to strings
data=data.astype(str)
print(data.dtypes)

Learner SignUp DateTime    object
Opportunity Id             object
Opportunity Name           object
Opportunity Category       object
Opportunity End Date       object
First Name                 object
Date of Birth              object
Gender                     object
Country                    object
Institution Name           object
Current/Intended Major     object
Entry created at           object
Status Description         object
Status Code                object
Apply Date                 object
Opportunity Start Date     object
dtype: object


In [8]:
# Convert only the true date columns to datetime
date_columns = [
    "Learner SignUp DateTime",
    "Opportunity End Date",
    "Date of Birth",
    "Entry created at",
    "Apply Date",
    "Opportunity Start Date"
]

for col in date_columns:
    data[col] = pd.to_datetime(data[col], format='mixed', errors='coerce')

#cp1["Date of Birth"] = pd.to_datetime(cp1["Date of Birth"], format='mixed', errors='coerce').dt.date


print(data.dtypes)
print(data.shape)



Learner SignUp DateTime    datetime64[ns]
Opportunity Id                     object
Opportunity Name                   object
Opportunity Category               object
Opportunity End Date       datetime64[ns]
First Name                         object
Date of Birth              datetime64[ns]
Gender                             object
Country                            object
Institution Name                   object
Current/Intended Major             object
Entry created at           datetime64[ns]
Status Description                 object
Status Code                        object
Apply Date                 datetime64[ns]
Opportunity Start Date     datetime64[ns]
dtype: object
(8548, 16)


***Handling values errors ***
- some values are written inconsistently

In [9]:
# getting the columns with issues
#print(data['Country'].unique())
#print(data['Institution Name'].unique())

In [10]:
#Handling Inconsistent country names based on unique vallues details:
data['Country'] = data['Country'].str.replace('U.S.', 'United States')
data['Country'] = data['Country'].str.replace('Iran, Islamic Republic of Persian Gulf', 'Iran')
data['Country'] = data['Country'].str.replace('Iran  Islamic Republic of Persian Gulf', 'Iran')

#Handling Inconsistent Institution names based on unique vallues details:
data['Institution Name'] = data['Institution Name'].str.replace('saint louis university', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].str.replace('Saint Louis university', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].str.replace('Saint louis university', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].str.replace('SAINT LOUIS UNIVERSITY', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].str.replace('Saint louis University', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].str.replace('St Louis University', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].str.replace('Saint Louis', 'Saint Louis University')
# noticed the name coming as Saint Louis University University so tried to adjust
data['Institution Name'] = data['Institution Name'].str.replace('Saint Louis University University', 'Saint Louis')

data['Institution Name'] = data['Institution Name'].str.replace('saint Louis University', 'Saint Louis')
data['Institution Name'] = data['Institution Name'].replace('Saint Louis', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].replace('saint Louis university', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].replace('saint louis', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].replace('saint louis University', 'Saint Louis University')
data['Institution Name'] = data['Institution Name'].replace('SAINT LOUIS', 'Saint Louis University')
print(data['Institution Name'].value_counts())
  


Institution Name
Saint Louis University                                4321
Illinois Institute of Technology                       103
Webster University                                      46
University of Ibadan                                    26
Kwame Nkrumah University of Science and Technology      22
                                                      ... 
Federal University of Petroleum Resources  Effurun       1
ADDIS ABABA INSTITUTE OF TECHNOLOGY                      1
Marwadi University                                       1
RMKCET                                                   1
NIIT  Accra                                              1
Name: count, Length: 2073, dtype: int64


In [11]:
# converting DOB to disregard time
data["Date of Birth"] = pd.to_datetime(data["Date of Birth"], format='mixed', errors='coerce').dt.date
#inputting place holder to avoid dropping rows
data['Opportunity Start Date']=data['Opportunity Start Date'].fillna('NA')


In [12]:
# Drop null rows 
data_cleaned = data
data_cleaned = data_cleaned.dropna()
data_cleaned.shape


(6839, 16)

**Feature Engineering**


In [13]:
#reading and viewing the cleaned file
#cleaned_data=pd.read_excel(output_path0)
data_cleaned.head()

,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date
0,2023-06-14 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Faria,2001-12-01,Female,Pakistan,Nwihs,Radiology,2024-11-03 12:01:41,Started,1080,2023-06-14 12:36:09,2022-03-11 18:30:39
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Poojitha,2000-08-16,Female,India,Saint Louis University,Information Systems,2024-11-03 12:01:41,Started,1080,2023-01-05 06:08:21,2022-03-11 18:30:39
3,2023-08-29 05:20:03,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Amrutha Varshini,1999-01-11,Female,United States,Saint Louis University,Information Systems,2024-11-03 12:01:41,Team Allocated,1070,2023-09-10 22:02:42,2022-03-11 18:30:39
4,2023-06-01 15:26:36,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Vinay Varshith,2000-04-19,Male,United States,Saint Louis University,Computer Science,2024-11-03 12:01:41,Started,1080,2023-06-01 15:40:10,2022-03-11 18:30:39
5,2024-02-03 19:16:07,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Mor,1996-12-05,Male,India,Saint Louis University,Mechanical Engineering,2024-11-03 12:01:41,Waitlisted,1040,2024-02-03 20:30:35,NA


In [14]:
#cleaned_data['Current/Intended Major'].unique()
cleaned_data = data_cleaned

In [15]:
#creating Major Category (STEM/non-STEM/Others) using unique values
stem=['Radiology', 'Information Systems', 'Computer Science', 'Epidemiology', 'Petroleum Engineering', 
       'Mechanical Engineering', 'Computer Science and Engineering','Pre Engg', 'Civil and Structural Engineering',
       'Artificial Intelligence', 'Robotics and Automation Engineering','Program in Exploratory Studies', 'Architectural Engineering',
       'Data Visualization', 'Public Health','Biotechnology', 'Airline Pilot', 'Food Safety and Technology','Medical Laboratory Science',
       'Systems Engineering','Geology', 'Neuroscience', 'Data Analysis', 'Bsc Nursing', 'Information Engineering','Electronics and Computer Engineering',
       'Architecture', 'Computer Science and Information Systems','Aeronautics', 'Engineers Physics', 'Construction Engineering and Management',
       'Soil Science and Technology', 'Business','Bs Forestry and Wildlife Management', 'Medical Laboratory','Occupational Therapy', 
       'Analytics', 'Biology', 'Economics', 'Mathematics', 'Bioinformatics','Data Science and Artificial Intelligence','Aeronautical Engineering',
       'Msc cyber forensics and security', 'Statistics and Operational Research','Master in VLSI and Microelectronics',
        'Biomedical Engineering', 'Data Science', 'Statistics','Economics and Econometrics', 'Computer and Infromation Sciences',
       'Bsc in Computer Science', 'Electronics and Communication', 'Computer Information Systems','Post Graduate Diploma in Public Health',
       'Electrical and Electronic Engineering', 'Electrical and Computer Engineering','Management Information Systems', 'Project Management',
       'Medicine', 'Information', 'Information Technology', 'Monitoring and Evaluation', 'Electrical Engineer','Social Development and Policy',
       'Actuarial Mathematics', 'Software Engineering', 'Linguistics', 'Voice Performance', 'Bachelor of Laws', 'Automotive Engineering',
       'Library and Information Management', 'Biological Sciences', 'Urban and Housing Development', 'Cyber Security', 'Data Analytics',
       'Computer Engineering', 'Environmental Sciences','Industrial Engineering', 'Business Analytics','Functional Engineering',
       'Artificial Intelligence and Machine learning','Forensic', 'Forensic Science', 'SUPPLY CHAIN MANAGEMENT',
       'Agriculture and Forestry', 'Pure and Applied Physics','Rehabilitation and Mental Health Counseling','MS in Information Science Machine Learning',
      'Applied Geophysics','Electronics and Computers Engineering', 'Computer and Information Sciences', 'B TECH', 'Accountancy',
       'Information technology Management', 'Archaeology', 'Logistics', 'ENGINEERING Management', 'MASTERS in INFORMATION SYSTEMS',
       'Bnsc Nursing Science', 'Computer Science Engineering', 'Aviation Management', 'Land Economy', 'ACCOUNTING',
       'MSc in Computational Decision Science and Operations Research', 'Biochemistry', 'Information Technology and Management',
       'Civil Engineering', 'Electrical Engineering', 'Cybersecurity', 'Computer Science with Internet of Things',
       'Pharmacy and Pharmacology', 'Nutrition Science', 'Metallurgy and Materials Engineering', 'Finance', 'Geography',
       'Information technology and management', 'Dentistry', 'Diploma in Visual Effects', 'Digital Media', 'Computer and Information Sciencess',
       'Environmental Sustainability', 'Electronics and Computer Science','MSc in Information technology','Tech and Business Management',
       'Communication and Software Engineer', 'Respiratory Therapy','Health Information Management', 'Veterinary Science',
       'Health Education','Food Science', 'Psychology', 'MS in Supply Chain Management', 'Financial Technology', 'Information technology', 
'Health Data Science', 'Electronics and Communication Engineering', 'Quantity Surveying','Masters in Information Systems', 
       'Masters in Computer Science','Environmental and Safety Engineering','ESports and Game Development', 'Could Computing',
       'Network and Information Security', 'MBA','Computer Applications', 'Mechanical and Aerospace Engineering',
       'Mechatronics Engineering', 'Supply Chain Management', 'Nursing','Animal Production and Health','Artificial Intelligence and Robotics',
       'Chemistry','Btech','Chemical Engineering',  'Physics and Astronomy','Earth and Marine Sciences', 'ANALYTICS',
       'Radiation Therapy',  'Engineering','Aerospace Engineering', 'Bachelor of Engineering', 'Information Science', 
       'Associate of Science', 'Marketing Analytics', 'Medical Sciences','Finance and Banking', 'Agricultural Extension and Education',
       'Applied Financial Economics', 'DOCTOR of PHARMACY', 'Bsc Agri', 'Telecommunication Engineering','Computing', 
       'Anatomy and Physiology','Artificial Intelligence and Data Science','Electronics and Computer Science Engineering',
       'Physics','MS Biostatistics and Health Analytics','Computer Science and Business Systems',  'Computer', 'Manufacturing Engineering',
       'Diploma in Automation and Robotics', 'Archeology and Cinematography', 'Carpentry and Joinery','Data Analyst','B Tech', 
      'Computer science engineering', 'Ms in Analytics','Food Process Engineering','Data Sceince', 'Animal Production',
       'COMPUTER SCIENCE and DESIGN', 'Business Informatics','Cyber Forensics and Security', 'Management Science and Analytics',
       'Actuarial Science', 'Mineral and Mining Engineering', 'People Analytics', 'Autonomous Systems and Robotics', 'Nurse',
       'Business Education', 'Data science', 'Remote Sensing','IT',  'Electronics and Telecommunication Engineering', 'MECHANICAL',
       'Textile Engineering', 'Machine Learning', 'Analysis','Healthcare Management','Physical Therapy', 'Human Nutrition and Dietetics',
       'Health Sciences', 'Microbiology', 'Information Studies', 'Physiotherapy', 'Statistics and Computeri Science', 'Radiography', 
       'MS in Cybersecurity', 'Msc in Disaster Management and Climate Sustainability Studies','Cyber Security and Digital Forensics', 
       'Procurement and Supply Chain Management', 'Information systems','Fisheries',  'Computer Science and Design', 'BSC Cs',
       'Msters in Healthcare Administration','Computer Networking and IT Security','Health Informatics','Bachelor Degree in Nursing',
       'Geoinformatics and Geospatial Analytics', 'Informatics','Electronics and Instrumentation',
       'Bachelor of Computer', 'Accounting', 'ECONOMICS and FINANCE','Geographic Information Science',
       'Agricultural Economics and Farm Management', 'General Science', 'Working As Software Devloper',]

nstem=['Business Administration','Business and Management Studies','Accounting and Finance','Human Resource', 'Law','Journalism',
       'Secretarial','Human Resources',  'Philosophy','Entrepreneurship','Communication and Media Studies', 'BCOM International Business', 
       'Law and Legal Studies', 'Theology or Divinity and Religious Studies', 'International Business Management','Advertising', 'History',
       'Politics', 'Marketing', 'English', 'English Language and Literature','Communication','English Linguistics','Social Policy and Administration',
       'Hospitality and Leisure Management', 'Mass Communication', 'Commerce','Digital Marketing','Political Science', 'Not Know Ab',
       'International Relations',  'Public Administration', 'Education and Training','Sociology', 'Business Chinese', 'Music', 'Education Management',
          'Management', 'Procurement', 'Art and Design','Development Studies', 'General Arts','Medical Marketing or Medical Administration',
     'Guidance and Counseling', 'Arts', 'Home Economics', 'Performing Arts','Textile', 'Education and Geography','General Management',
        'Development Economics', 'Film Making','International Development',  'International Business', 'Communication and Language Arts Education',
       'Early Childhood Education','Islamic Studies', 'Multimedia Communication','Entrepreneurship Studies','Advertising and Packaging',
       'American Sign Language Interpretation', 'Communications', 'Brand Strategy', 'Leadership and Organizational development',
       'Education Administration',]

others=['Oth','I Am Major','Other', 'Student', 'Eee', 'Social Work','Pol','Fresher','Faizan','Degree','BSIT','Job', 'Leadership','Gamer', 'Ot',
        'BBA','MCA',  'No', 'Aa','Community Health Officer', 'Social S','Ba Psych', 'Shaista', 'Ha', 'Unemployed','Dropped Out',
       'After Bachelor', 'MMG', 'Goodcreativity','Bachelor','Diploma','BCA', 'TE','Manager','Yoganand Sir', 'High School Certificate', 'BAF',
       'Lower Credit','Certification','Pursuing MCA','Othe', 'Honours', 'ISE','Otheraassss','High School', 'To Study', 'General Practitioner',
       'Research','Non', 'Sdada', 'Ghj', 'Na', 'General', 'Bcom','Masters', 'B Sc','Www','Already Graduation Completed','GENERAL MANAGEMENT', 
       'High School Student', 'Pv','Home Tutor', 'Cycw', 'Nil','Pos Service']
#creating empty column
data_cleaned['major_category']=''
# assigning conditions
data_cleaned.loc[data_cleaned['Current/Intended Major'].isin(stem), 'major_category']='STEM'
data_cleaned.loc[data_cleaned['Current/Intended Major'].isin(nstem), 'major_category']='non-STEM'
data_cleaned.loc[data_cleaned['Current/Intended Major'].isin(others), 'major_category']='Others'


C:\Users\navne\AppData\Local\Temp\ipykernel_17816\4194128610.py:74: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['major_category']=''


In [16]:
data_cleaned.Country.unique()

array(['Pakistan', 'India', 'United States', 'United Arab Emirates',
       'Nigeria', 'Egypt', 'Nepal', 'Kenya', 'Ghana', 'Zambia', 'Morocco',
       'Ethiopia', 'Zimbabwe', 'Uganda', 'Indonesia', 'Cameroon', 'Yemen',
       'China', 'Bangladesh', 'Congo', 'Liberia', 'United Kingdom',
       'Vietnam', 'Japan', 'Rwanda', 'Gambia', 'Philippines', 'Australia',
       'Somalia', 'Sierra Leone', 'Lebanon', 'Botswana', 'Iraq',
       'Uzbekistan', 'Turkey', 'Honduras',
       'Tanzania, United Republic of Tanzania',
       'British Indian Ocean Territory', 'France', 'Belarus', 'Algeria',
       'Korea, Republic of South Korea', 'Mauritius', 'Tunisia',
       'Kazakhstan', 'Peru', 'Brazil', 'Ukraine', 'South Africa',
       'Germany', 'Namibia', 'Iran', 'American Samoa',
       'Falkland Islands (Malvinas)', 'Saudi Arabia', 'Azerbaijan',
       'Dominican Republic', 'Lesotho', 'Malaysia', 'Singapore',
       'Libyan Arab Jamahiriya', "Cote d'Ivoire", 'Afghanistan', 'Canada',
       'Bhutan'

In [17]:
#creating country regions countries unique values 
Africa=['Algeria', 'Botswana', 'Cameroon', 'Congo', "Cote d'Ivoire", 'Egypt', 'Ethiopia', 'Gambia', 'Ghana', 'Kenya', 'Lesotho', 
'Liberia', 'Libyan Arab Jamahiriya', 'Mauritius', 'Morocco', 'Namibia', 'Nigeria', 'Rwanda', 'Sierra Leone', 'Somalia',
 'South Africa', 'Tanzania, United Republic of Tanzania', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe']
Asia=['Afghanistan', 'Azerbaijan', 'Bangladesh', 'Bhutan', 'British Indian Ocean Territory', 'China', 'India', 'Indonesia', 'Iran',
 'Iraq', 'Japan', 'Kazakhstan', 'Korea, Republic of South Korea', 'Lebanon', 'Malaysia', 'Nepal', 'Pakistan', 'Philippines',
 'Saudi Arabia', 'Singapore', 'Turkey', 'United Arab Emirates', 'Uzbekistan', 'Vietnam', 'Yemen']
Europe=['Belarus', 'France', 'Germany', 'Spain', 'Ukraine', 'United Kingdom']
North_America=['Canada', 'Dominican Republic', 'Honduras', 'United States']
Oceania=['American Samoa', 'Australia']
South_America=['Brazil', 'Falkland Islands (Malvinas)', 'Peru']
#creating empty column
data_cleaned['region']=''
# assigning conditions
data_cleaned.loc[data_cleaned['Country'].isin(Africa), 'region']='Africa'
data_cleaned.loc[data_cleaned['Country'].isin(Asia), 'region']='Asia'
data_cleaned.loc[data_cleaned['Country'].isin(Europe), 'region']='Europe'
data_cleaned.loc[data_cleaned['Country'].isin(North_America), 'region']='North America'
data_cleaned.loc[data_cleaned['Country'].isin(South_America), 'region']='South_America'
data_cleaned.loc[data_cleaned['Country'].isin(Oceania), 'region']='Oceania'


C:\Users\navne\AppData\Local\Temp\ipykernel_17816\1428837991.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['region']=''


In [18]:
# replacing all but one university due to its high count
university_to_keep = 'Saint Louis University'

# Assign the new value 'Others' to other rows
data_cleaned.loc[data_cleaned['Institution Name'] != university_to_keep, 'Institution Name'] = 'Others'

print("\nDataFrame after replacement:")
data_cleaned.head()


DataFrame after replacement:


,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date,major_category,region
0,2023-06-14 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Faria,2001-12-01,Female,Pakistan,Others,Radiology,2024-11-03 12:01:41,Started,1080,2023-06-14 12:36:09,2022-03-11 18:30:39,STEM,Asia
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Poojitha,2000-08-16,Female,India,Saint Louis University,Information Systems,2024-11-03 12:01:41,Started,1080,2023-01-05 06:08:21,2022-03-11 18:30:39,STEM,Asia
3,2023-08-29 05:20:03,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Amrutha Varshini,1999-01-11,Female,United States,Saint Louis University,Information Systems,2024-11-03 12:01:41,Team Allocated,1070,2023-09-10 22:02:42,2022-03-11 18:30:39,STEM,North America
4,2023-06-01 15:26:36,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Vinay Varshith,2000-04-19,Male,United States,Saint Louis University,Computer Science,2024-11-03 12:01:41,Started,1080,2023-06-01 15:40:10,2022-03-11 18:30:39,STEM,North America
5,2024-02-03 19:16:07,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Mor,1996-12-05,Male,India,Saint Louis University,Mechanical Engineering,2024-11-03 12:01:41,Waitlisted,1040,2024-02-03 20:30:35,NA,STEM,Asia


In [19]:
data_cleaned['Status Description'].unique()

array(['Started', 'Team Allocated', 'Waitlisted', 'Withdraw',
       'Rewards Award', 'Dropped Out', 'Rejected', 'Applied'],
      dtype=object)

In [20]:
#creating dropped out column
dropped=['Dropped Out']
not_dropped=['Rewards Award','Withdraw']
#creating empty column
data_cleaned=data_cleaned.assign(dropped_out=None)
# assigning conditions
data_cleaned.loc[data_cleaned['Status Description'].isin(dropped), 'dropped_out']='Yes'
data_cleaned.loc[data_cleaned['Status Description'].isin(not_dropped), 'dropped_out']='No'
data_cleaned.head()

,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date,major_category,region,dropped_out
0,2023-06-14 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Faria,2001-12-01,Female,Pakistan,Others,Radiology,2024-11-03 12:01:41,Started,1080,2023-06-14 12:36:09,2022-03-11 18:30:39,STEM,Asia,None
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Poojitha,2000-08-16,Female,India,Saint Louis University,Information Systems,2024-11-03 12:01:41,Started,1080,2023-01-05 06:08:21,2022-03-11 18:30:39,STEM,Asia,None
3,2023-08-29 05:20:03,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Amrutha Varshini,1999-01-11,Female,United States,Saint Louis University,Information Systems,2024-11-03 12:01:41,Team Allocated,1070,2023-09-10 22:02:42,2022-03-11 18:30:39,STEM,North America,None
4,2023-06-01 15:26:36,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Vinay Varshith,2000-04-19,Male,United States,Saint Louis University,Computer Science,2024-11-03 12:01:41,Started,1080,2023-06-01 15:40:10,2022-03-11 18:30:39,STEM,North America,None
5,2024-02-03 19:16:07,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Mor,1996-12-05,Male,India,Saint Louis University,Mechanical Engineering,2024-11-03 12:01:41,Waitlisted,1040,2024-02-03 20:30:35,NA,STEM,Asia,None


In [21]:
# Ensure DOB is a pandas Timestamp (not python date) and compute age safely
data_cleaned['Date of Birth'] = pd.to_datetime(data_cleaned['Date of Birth'], errors='coerce')

today = pd.Timestamp.today().normalize()
new_age = (today - data_cleaned['Date of Birth']).dt.days // 365

# If an 'age' column already exists, keep it where DOB is missing; otherwise use new_age
if 'age' in data_cleaned.columns:
    data_cleaned['age'] = new_age.fillna(data_cleaned['age'])
else:
    data_cleaned['age'] = new_age

data_cleaned[['Date of Birth', 'age']].head()

,Date of Birth,age
0,2001-12-01,23
1,2000-08-16,25
3,1999-01-11,26
4,2000-04-19,25
5,1996-12-05,28


In [22]:
#creating signup year, month and weekday
# Extract month/year from Learner SignUp DateTime and compute time in opportunity
data_cleaned['Learner SignUp DateTime'] = pd.to_datetime(data_cleaned.get('Learner SignUp DateTime'), errors='coerce')
if 'Learner SignUp DateTime' in data_cleaned.columns:
    data_cleaned['signup_month'] = data_cleaned['Learner SignUp DateTime'].dt.month
    data_cleaned['signup_year'] = data_cleaned['Learner SignUp DateTime'].dt.year
    data_cleaned['signup_weekday'] = data_cleaned['Learner SignUp DateTime'].dt.weekday
    print('Added signup_weekday, signup_month and signup_year')



Added signup_weekday, signup_month and signup_year


In [23]:
# Make sure date columns are proper datetimes, then compute engagement_days/time_in_opportunity safely
for col in ['Opportunity Start Date', 'Apply Date', 'Opportunity End Date']:
    data_cleaned[col] = pd.to_datetime(data_cleaned.get(col), errors='coerce')

# engagement_days = (Start - Apply) in days; keep only non-negative values (NaN when missing or Start < Apply)
data_cleaned['engagement_days'] = (data_cleaned['Opportunity Start Date'] - data_cleaned['Apply Date']).dt.days
data_cleaned['engagement_days'] = data_cleaned['engagement_days'].where(data_cleaned['engagement_days'] >= 0, pd.NA)

# time in opportunity = End - Start in days (may be negative if data error; we'll keep as-is for now to review)
data_cleaned['time_in_opportunity'] = (data_cleaned['Opportunity End Date'] - data_cleaned['Opportunity Start Date']).dt.days

# Quick checks
print('engagement_days — missing:', data_cleaned['engagement_days'].isna().sum())
print('time_in_opportunity — missing:', data_cleaned['time_in_opportunity'].isna().sum())
data_cleaned[['Opportunity Start Date','Apply Date','Opportunity End Date','engagement_days','time_in_opportunity']].head()

engagement_days — missing: 5634
time_in_opportunity — missing: 3554


,Opportunity Start Date,Apply Date,Opportunity End Date,engagement_days,time_in_opportunity
0,2022-03-11 18:30:39,2023-06-14 12:36:09,2024-06-29 18:52:39,NaN,841.0
1,2022-03-11 18:30:39,2023-01-05 06:08:21,2024-06-29 18:52:39,NaN,841.0
3,2022-03-11 18:30:39,2023-09-10 22:02:42,2024-06-29 18:52:39,NaN,841.0
4,2022-03-11 18:30:39,2023-06-01 15:40:10,2024-06-29 18:52:39,NaN,841.0
5,NaT,2024-02-03 20:30:35,2024-06-29 18:52:39,NaN,NaN


In [24]:
# Export the dataframe to an Excel file
output_path = r"C:\Users\maruf\Feature_Engineered_Data.xlsx"

# index=False → removes the dataframe index column
# engine='openpyxl' ensures correct Excel writing
cleaned_data.to_excel(output_path, index=False, engine='openpyxl')

print(f"✅ DataFrame exported successfully to '{output_path}'")
cleaned_data.shape

OSError: Cannot save file into a non-existent directory: 'C:\Users\maruf'

**Splitting dataset for training and testing**

- Normalize numerical features  data (age, engagement_days, time_in_opportunity)

In [25]:
from sklearn.preprocessing import StandardScaler
num_cols = [c for c in ['age','engagement_days','time_in_opportunity'] if c in data_cleaned.columns]
if num_cols:
    scaler = StandardScaler()
    data_cleaned[num_cols] = scaler.fit_transform(data_cleaned[num_cols].fillna(0))
    print('Normalized:', num_cols)
    

Normalized: ['age', 'engagement_days', 'time_in_opportunity']


In [27]:
#dropped rows with missing dropoff data(with status-started, waitlisted, team allocated, and rejected)
df=data_cleaned.dropna(subset=['dropped_out'])
df.shape

(676, 25)

In [28]:
#selecting predicting target
y=df.dropped_out
#choosing features
features=['Opportunity Category','Gender','Status Code','major_category','age','signup_month','signup_year','Institution Name',
          'engagement_days','time_in_opportunity','signup_weekday','region']
X=df[features]
#splitting data
from sklearn.model_selection import train_test_split
train_X,test_X,train_y,test_y=train_test_split(X,y, test_size=0.4, random_state=1)

df.shape

(676, 25)

- One-hot encode categorical columns for training data: Gender, Country, Opportunity Category,major_category, region, dropped_out

In [29]:
cat_cols = [c for c in ['Gender','Opportunity Category','major_category','region','Institution Name'] if c in train_X.columns]
from sklearn.preprocessing import OneHotEncoder

# Create an instance of OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore')

# Fit the encoder on the training data only
ohe.fit(train_X[cat_cols])

# Transform training data
X_train_encoded = ohe.transform(train_X[cat_cols])

# Get the new column names created by the encoder
ohe_feature_names = ohe.get_feature_names_out(input_features=cat_cols)

# Create dataframes from the encoded sparse matrices
X_train_ohe = pd.DataFrame(X_train_encoded.toarray(), columns=ohe_feature_names, index=train_X.index)

# Drop the original categorical columns from the dataframes
train_X = train_X.drop(columns=cat_cols)

# Concatenate the original and new encoded dataframes
train_X = pd.concat([train_X, X_train_ohe], axis=1)

# Transform test data
X_test_encoded = ohe.transform(test_X[cat_cols])
# Create dataframes from the encoded sparse matrices
X_test_ohe = pd.DataFrame(X_test_encoded.toarray(), columns=ohe_feature_names, index=test_X.index)
# Drop the original categorical columns from the dataframes
test_X = test_X.drop(columns=cat_cols)
# Concatenate the original and new encoded dataframes
test_X = pd.concat([test_X, X_test_ohe], axis=1)
test_X

,Status Code,age,signup_month,signup_year,engagement_days,time_in_opportunity,signup_weekday,Gender_Don't want to specify,Gender_Female,Gender_Male,...,major_category_STEM,major_category_non-STEM,region_Africa,region_Asia,region_Europe,region_North America,region_Oceania,region_South_America,Institution Name_Others,Institution Name_Saint Louis University
1744,1050,-0.118277,12,2023,2.144784,-0.373702,2,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1982,1050,0.112178,12,2023,2.100042,-0.373702,5,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2840,1050,0.573086,3,2023,-0.360736,-0.648892,6,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4214,1050,-0.579185,6,2023,2.234267,-0.373702,2,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4195,1110,-0.579185,6,2023,-0.360736,-0.648892,4,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4135,1050,0.342632,7,2024,-0.360736,-0.648892,0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4920,1050,-1.270548,1,2023,2.894203,-0.373702,1,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3339,1050,-0.809639,12,2023,0.992692,-0.373702,3,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2978,1050,-0.118277,12,2023,2.077672,-0.373702,1,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [31]:

# Export the  testing target to an Excel file
output_path1a =  r"test_y.xlsx"
output_path1b =  r"train_y.xlsx"
output_path2a =  r"testX_Data.xlsx"
output_path2b=  r"trainX_Data.xlsx"

# index=False → removes the dataframe index column
# engine='openpyxl' ensures correct Excel writing
test_y.to_excel(output_path1a, index=False, engine='openpyxl')
train_y.to_excel(output_path1b, index=False, engine='openpyxl')
test_X.to_excel(output_path2a, index=False, engine='openpyxl')
train_X.to_excel(output_path2b, index=False, engine='openpyxl')

print(f"✅ Model test target and features '{output_path1a}', '{output_path2a}'")
print(f"✅ Model train target and features to '{output_path1b}', '{output_path2b}'")

✅ Model test target and features 'test_y.xlsx', 'testX_Data.xlsx'
✅ Model train target and features to 'train_y.xlsx', 'trainX_Data.xlsx'


In [32]:
# Cross-validation (5-fold) with a Random Forest and quick test-set evaluation
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Prepare X/y (fill NaNs defensively)
X = train_X.fillna(0)
y = train_y.astype(str).fillna("missing")  # ensure no NA strings

# Encode labels if necessary
le = LabelEncoder()
y_enc = le.fit_transform(y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

acc = cross_val_score(model, X, y_enc, cv=cv, scoring="accuracy", n_jobs=-1)
roc = cross_val_score(model, X, y_enc, cv=cv, scoring="roc_auc", n_jobs=-1)

print(f"CV Accuracy (5-fold): {acc.mean():.3f} ± {acc.std():.3f}")
print(f"CV ROC AUC  (5-fold): {roc.mean():.3f} ± {roc.std():.3f}")

# Fit on training data and evaluate on the hold-out test set
model.fit(X, y_enc)
X_test = test_X.fillna(0)
y_test = test_y.astype(str).fillna("missing")
y_test_enc = le.transform(y_test)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nHold-out test results:")
print("Accuracy:", accuracy_score(y_test_enc, y_pred))
print("ROC AUC:", roc_auc_score(y_test_enc, y_proba))
print("\nClassification report:\n", classification_report(y_test_enc, y_pred, target_names=le.classes_))

ModuleNotFoundError: No module named '_posixsubprocess'

In [33]:
df.dtypes

Learner SignUp DateTime    datetime64[ns]
Opportunity Id                     object
Opportunity Name                   object
Opportunity Category               object
Opportunity End Date       datetime64[ns]
First Name                         object
Date of Birth              datetime64[ns]
Gender                             object
Country                            object
Institution Name                   object
Current/Intended Major             object
Entry created at           datetime64[ns]
Status Description                 object
Status Code                        object
Apply Date                 datetime64[ns]
Opportunity Start Date     datetime64[ns]
major_category                     object
region                             object
dropped_out                        object
age                               float64
signup_month                        int32
signup_year                         int32
signup_weekday                      int32
engagement_days                   

## Recomendation System

In [34]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

df_unique = df.drop_duplicates(subset='Opportunity Id').reset_index(drop=True)

# Select relevant numerical features for similarity
features = ['signup_month', 'signup_year', 'signup_weekday', 
            'engagement_days', 'time_in_opportunity', 'age']

# Fill missing values and scale features
X = df_unique[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [35]:
# Compute cosine similarity between opportunities
similarity_matrix = cosine_similarity(X_scaled)

# Build DataFrame for easy lookup
similarity_df = pd.DataFrame(
    similarity_matrix, 
    index=df_unique['Opportunity Id'], 
    columns=df_unique['Opportunity Id']
)

print("✅ Similarity matrix ready:", similarity_df.shape)


✅ Similarity matrix ready: (21, 21)


In [36]:
# Function to recommend similar opportunities
def recommend_similar_opportunities(opportunity_id, top_n=5):
    if opportunity_id not in similarity_df.index:
        print(" Opportunity ID not found!")
        return None
    
    # Get top-N most similar opportunities (skip itself)
    similar_scores = similarity_df.loc[opportunity_id].sort_values(ascending=False).iloc[1:top_n+1]
    
    # Collect info about those similar opportunities
    cols_to_show = ['Opportunity Id', 'Opportunity Name', 'Opportunity Category', 
                    'region', 'major_category']
    if 'dropped_out' in df_unique.columns:
        cols_to_show.append('dropped_out')

    similar_opp = df_unique.loc[df_unique['Opportunity Id'].isin(similar_scores.index), cols_to_show].copy()
    similar_opp['similarity_score'] = similar_scores.values
    return similar_opp

# Example: get recommendations for one opportunity
example_id = df_unique['Opportunity Id'].iloc[3]
recommendations = recommend_similar_opportunities(example_id, top_n=5)

print(f"Opportunities similar to: {example_id}")
display(recommendations)


Opportunities similar to: 00000000-0GZY-NNHV-FJWW-PTA9VX


,Opportunity Id,Opportunity Name,Opportunity Category,region,major_category,dropped_out,similarity_score
3,00000000-0GZY-NNHV-FJWW-PTA9VX,Innovation & Entrepreneurship,Internship,Asia,STEM,None,1.000000
6,00000000-10WC-BS50-CYGD-X97ES4,CPR/AED Certification,Course,North America,STEM,None,0.951961
12,00000000-101Y-HSX2-0DFJ-QCKQBR,AI Ethics Challenge,Competition,North America,STEM,None,0.686418
14,00000000-10CD-3CAK-3FZK-9VXKRM,Digital Strategy Virtual Internship,Internship,North America,STEM,None,0.686418
15,00000000-101B-RJY9-TVWC-6XG9BH,Business Consulting,Internship,North America,STEM,None,0.437836


### Random-Forest-Based Recommendation
Use the earlier trained random forest as a ranking signal instead of pure similarity.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
df = data_cleaned.copy()
features = ['Opportunity Category','Gender','Status Code','major_category','age',
            'signup_month','signup_year','Institution Name','engagement_days',
            'time_in_opportunity','signup_weekday','region']
cat_cols = ['Opportunity Category','Gender','Status Code','major_category','Institution Name','region']
rf_df = df.dropna(subset=['dropped_out']).reset_index(drop=True)
X_raw = rf_df[features].copy()
X_raw[cat_cols] = X_raw[cat_cols].fillna('missing')
encoder = OneHotEncoder(handle_unknown='ignore')
X_cat = encoder.fit_transform(X_raw[cat_cols])
X_cat = pd.DataFrame(
    X_cat.toarray(),
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_raw.index
)
X_full = pd.concat([X_raw.drop(columns=cat_cols), X_cat], axis=1).fillna(0)
target_encoder = LabelEncoder()
y_rf = target_encoder.fit_transform(rf_df['dropped_out'].astype(str))
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_full, y_rf)
stay_label = 'No'
classes = list(target_encoder.classes_)
if stay_label not in classes:
    stay_label = classes[0]
stay_class_index = classes.index(stay_label)
rf_df['stay_prob'] = rf_model.predict_proba(X_full)[:, stay_class_index]
rf_df['dropout_risk'] = 1 - rf_df['stay_prob']

def recommend_by_rf(top_n=5, keep_category=None):
    candidates = rf_df.copy()
    if keep_category is not None:
        candidates = candidates[candidates['Opportunity Category'] == keep_category]
    score_cols = [
        'Opportunity Id','Opportunity Name','Opportunity Category',
        'region','major_category','stay_prob','dropout_risk'
    ]
    return candidates.sort_values('stay_prob', ascending=False)[score_cols].head(top_n)
display(recommend_by_rf(top_n=5))

,Opportunity Id,Opportunity Name,Opportunity Category,region,major_category,stay_prob,dropout_risk
4,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,Asia,STEM,1.0,0.0
675,00000000-100J-PM3A-0WJ8-5T68M5,Jump Start: Developing your Emotional Intellig...,Course,North America,STEM,1.0,0.0
0,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,Asia,STEM,1.0,0.0
1,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,North America,STEM,1.0,0.0
662,00000000-10GG-17ZM-AP4T-WQVBGQ,Data Visualization Associate,Internship,Asia,STEM,1.0,0.0
